In [1]:
## This script will genrate heatmap of all cells across different archetypes and generalists
## You can use this script to draw heatmap of gene expression across all archetypes in a sample before after ParTI, after reclustering or after mellon
## But please provide the right path where the cells barcode information of individual archetypes including generalist is present. 
## It just only needs the barcodes informations from individual archetypes and generalist of a sample.

In [3]:
library(stringr)
library(glue)
library(gplots)
library(tools)
library(dplyr)
library(RColorBrewer)

sample <- "SMC25-T"

# Please put the path of mellon analysis result folder of your specific sample.
mellon_path <- paste("../../pipeline_testing/mellon_analysis/",sample,sep="/")  
mellon_archetype_files <- list.files(path = mellon_path)
cell_files <- mellon_archetype_files[str_detect(mellon_archetype_files, ".csv$") & str_detect(mellon_archetype_files,"specialists_and_generalists")]

mellon_specialist_generalist <- list()
for(i in 1:length(cell_files)){
    data <- read.table(paste(mellon_path,cell_files[i],sep="/"),header=TRUE,sep="\t")
    data$Index <- as.numeric(sub(",.*", "", data$X.cells)) 
    data$cells <- sub("^[^,]*,\\s*", "", data$X.cells)
    data <- data[,-1]
    mellon_specialist_generalist[[i]] <- data
}

# Please put the path of the folder of your respective sample where its gene expression data is present
gene_exp_file <- list.files(path= gene_exp_path)
gene_exp_path <- paste("../../pipeline_testing/data/",sample,sep="/") 
TPM_gene_exp_file <- gene_exp_file[str_detect(gene_exp_file, ".csv$") & str_detect(gene_exp_file, "_TPM_mod")]
tpm_data <- read.csv(paste(gene_exp_path, TPM_gene_exp_file, sep="/"),row.names=1, check.names = FALSE)
genes <- read.table(paste(gene_exp_path, "genes.list",sep="/"))
colnames(genes) <- "genes"
head(tpm_data)

mellon_specialist_generalist_exp <- list()
for (j in 1:length(mellon_specialist_generalist)){
    mellon_specialist_generalist_exp[[j]] <- tpm_data[,match(mellon_specialist_generalist[[j]]$cells,colnames(tpm_data))]
}

specialist_generalist_data <- do.call("cbind",mellon_specialist_generalist_exp)
specialist_generalist_data <- specialist_generalist_data[rowSums(specialist_generalist_data)>=1,]

#calculate varience across rows
specialist_generalist_data$varience <- apply(specialist_generalist_data[,-1], 1, var)
min(specialist_generalist_data$varience)
max(specialist_generalist_data$varience)
heatmap_data <- specialist_generalist_data[specialist_generalist_data$varience>0.3,]
heatmap_data <- heatmap_data[,1:ncol(heatmap_data)-1]
print(paste("up_deg heatmap input data dimension",dim(heatmap_data)))
matrix_heatmap <- as.matrix(heatmap_data)

#Please mention your folder path where your heatmap image will be saved
export_path <- "../../pipeline_testing/heatmap_mellon_test"  
dir.create(export_path,recursive=T)

# ## Please mention the color for different specialists and generalist as per your choice
# pdf(file=paste(export_path,glue("{sample}_heatmap_specialist_&_generalist.pdf"),sep="/"),width=20,height=20)
# par(mar = c(1, 4, 2, 4))
# heatmap.2(matrix_heatmap,trace="none",Colv=FALSE,Rowv=TRUE,dendrogram='none',cexRow=1,cexCol=0.7,keysize=0.8,col=bluered(256), margins=c(25,12),
#           ColSideColors=c(rep("royalblue",ncol(mellon_specialist_generalist_exp[[1]])), rep("cyan",ncol(mellon_specialist_generalist_exp[[2]])), 
#                           rep('yellowgreen',ncol(mellon_specialist_generalist_exp[[3]])),rep('olivedrab',ncol(mellon_specialist_generalist_exp[[4]])), 
#                           rep('darkolivegreen',ncol(mellon_specialist_generalist_exp[[5]])),
#                           rep("seagreen",ncol(mellon_specialist_generalist_exp[[6]])),rep("rosybrown",ncol(mellon_specialist_generalist_exp[[7]]))),
#                           scale='row',breaks=seq(-2,2,length.out=257))

# legend("topright",legend=c("Archetype 1", "Archetype 2", "Archetype 3 cluster 0", "Archetype 3 cluster 1",
#                            "Archetype 3 cluster 2", "Archetype 4",  "Generalists"),
#        fill=c("royalblue", "cyan", 'yellowgreen', 'olivedrab', 'darkolivegreen', "seagreen", "rosybrown"), 
#        border=FALSE, bty="n", y.intersp = 0.7, cex=1.5)
# dev.off()

## Use this snippet if you want to put color automatically without any customization. While using this snippet please comment out line number 47-61
extract_filename_word <- function(filenames) {
  # 1. Remove file extension (.csv, .txt, etc.)
  clean_names <- file_path_sans_ext(filenames)
  
  # 2. Extract based on the conditions
  results <- ifelse(
    # Condition 1: If 'specialists' is present -> extract everything before it If 'generalist' is present
    str_detect(clean_names, regex("Archetype", ignore_case = TRUE)),
    sub("[_\\s]?specialists.*", "", clean_names, ignore.case = TRUE),
    
    # Condition 2: If 'generalist' is present
    ifelse(
      str_detect(clean_names, regex("generalist", ignore_case = TRUE)),
      str_extract(clean_names, regex("generalists?", ignore_case = TRUE)),
      NA_character_  
    )
  )
  return(results)
}
arch_names <- extract_filename_word(cell_files)

## Put the path where your sample wise consolidated data is present i.e. extracted from code8.
cons_data_dir <- "../../pipeline_testing/consolidated_data_samplewise/"  

mellon_data_files <- list.files(path = cons_data_dir, 
                                   pattern = "*.csv", 
                                   full.names = TRUE, 
                                   recursive = TRUE)

all_mellon_output <- lapply(mellon_data_files, function(file){
                        read.csv(file, row.names=1)
                    })

combined_df <- bind_rows(all_mellon_output)
combined_df <- combined_df %>%
                  mutate(
                    Arch_after_mellon = str_replace_all(Arch_after_mellon, "([a-zA-Z])([0-9])", "\\1_\\2"),
                    Arch_after_mellon = str_replace_all(Arch_after_mellon, "Generalist", "generalists")
                  )
all_arch_names <- sort(unique(combined_df$Arch_after_mellon))

all_unique_archetypes <- all_arch_names
n_clust <- length(all_unique_archetypes)

if (n_clust <= 8) {
  palette_pool <- brewer.pal(max(3, n_clust), "Set2")[1:n_clust]
} else if (n_clust <= 12) {
  palette_pool <- brewer.pal(n_clust, "Paired")
} else {
  # For > 12 clusters, interpolate a distinct palette
  palette_pool <- colorRampPalette(brewer.pal(12, "Paired"))(n_clust)
}

master_cluster_colors <- setNames(palette_pool, sort(all_unique_archetypes))
col_bar_colors <- master_cluster_colors[sort(arch_names)]

pdf(file=paste(export_path,glue("{sample}_heatmap_specialist_&_generalist.pdf"),sep="/"),width=20,height=20)
par(mar = c(1, 4, 2, 4))
heatmap.2(matrix_heatmap,trace="none",Colv=FALSE,Rowv=TRUE,dendrogram='none',cexRow=1,cexCol=0.7,keysize=0.8,col=bluered(256), margins=c(25,12),
          ColSideColors=c(rep(col_bar_colors[[1]],ncol(mellon_specialist_generalist_exp[[1]])), 
                          rep(col_bar_colors[[2]],ncol(mellon_specialist_generalist_exp[[2]])), 
                          rep(col_bar_colors[[3]],ncol(mellon_specialist_generalist_exp[[3]])),
                          rep(col_bar_colors[[4]],ncol(mellon_specialist_generalist_exp[[4]])), 
                          rep(col_bar_colors[[5]],ncol(mellon_specialist_generalist_exp[[5]])),
                          rep(col_bar_colors[[6]],ncol(mellon_specialist_generalist_exp[[6]])),
                          rep(col_bar_colors[[7]],ncol(mellon_specialist_generalist_exp[[7]]))),
                          scale='row',breaks=seq(-2,2,length.out=257))
legend("topright",legend= names(col_bar_colors),fill=col_bar_colors, border=FALSE, bty="n", y.intersp = 0.7, cex=1.5)
dev.off()

,SMC25-T_AAACGGGGTTATGTGC,SMC25-T_AAACGGGTCAACACAC,SMC25-T_AAAGATGGTAACGTTC,SMC25-T_AAATGCCAGTGATCGG,SMC25-T_AAATGCCTCAAACGGG,SMC25-T_AACCATGTCCTAAGTG,SMC25-T_AACTCAGCACCGAAAG,SMC25-T_AACTCAGGTAGAGTGC,SMC25-T_AACTCAGGTCCGAAGA,SMC25-T_AACTCAGGTTGGTAAA,⋯,SMC25-T_TTGCGTCGTTACGTCA,SMC25-T_TTGGAACCAATAACGA,SMC25-T_TTGGAACGTAAGTGGC,SMC25-T_TTGTAGGCAGTCAGAG,SMC25-T_TTTATGCTCGGCGGTT,SMC25-T_TTTCCTCAGCTGTTCA,SMC25-T_TTTGCGCCACACTGCG,SMC25-T_TTTGGTTAGCATCATC,SMC25-T_TTTGTCAAGCGTTCCG,SMC25-T_TTTGTCACAAATACAG
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,0.0000000,0,0,0,0,0,0.0000000,0,0.000000,0,⋯,0,0,0,1.620733,0,0,0,0,0,0
A1BG-AS1,0.0000000,0,0,0,0,0,0.0000000,0,0.000000,0,⋯,0,0,0,0.000000,0,0,0,0,0,0
A1CF,0.3651921,0,0,0,0,0,0.3118466,0,0.351024,0,⋯,0,0,0,0.000000,0,0,0,0,0,0
A2M,0.0000000,0,0,0,0,0,0.0000000,0,0.000000,0,⋯,0,0,0,0.000000,0,0,0,0,0,0
A2M-AS1,0.0000000,0,0,0,0,0,0.0000000,0,0.000000,0,⋯,0,0,0,0.000000,0,0,0,0,0,0
A2ML1,0.0000000,0,0,0,0,0,0.0000000,0,0.000000,0,⋯,0,0,0,0.000000,0,0,0,0,0,0


[1] 0.000151167

[1] 2.697839

[1] "up_deg heatmap input data dimension 1525"
[2] "up_deg heatmap input data dimension 1361"


Warning message in dir.create(export_path, recursive = T):
“'../../pipeline_testing/heatmap_mellon_test' already exists”
Warning message in heatmap.2(matrix_heatmap, trace = "none", Colv = FALSE, Rowv = TRUE, :
“Using scale="row" or scale="column" when breaks arespecified can produce unpredictable results.Please consider using only one or the other.”


agg_record_1278f7054a263 
                       2